# MACURA flies a Drone

A quadrotor learns to recover from a tumbling start, fly to a target, and hold a stable
hover. The high-angular-rate recovery is aggressive: a fixed-horizon rollout (MBPO)
over-imagines it and the drone destabilizes, while MACURA truncates the imagined rollout
when the model gets uncertain. We train the four algorithms (MACURA, MBPO, M2AC, SAC) -
shared SAC backbone and ensemble, only the rollout differs - and compare sample efficiency
and stability. Runs on a Kaggle GPU kernel, top to bottom.

## **Get the code**
Clone the project source (src/) from GitHub into /kaggle/working. Enable Internet in the
kernel settings and add a Secret named GITHUB_TOKEN (Add-ons -> Secrets) for the private repo.

In [ ]:
import os, sys, subprocess

# Bring the project source (src/) onto the Kaggle kernel by cloning it from GitHub.
# Needs: Internet ON (kernel settings) and a Kaggle Secret named GITHUB_TOKEN for the
# private repo. Everything lands in /kaggle/working, which Kaggle saves as the output.
REPO   = "silvergjeka22/macura-drone"
BRANCH = "main"
ROOT   = "/kaggle/working/macura-drone"          # where src/ is cloned

# read the GitHub token from a Kaggle Secret (or a GITHUB_TOKEN env var if you set one)
token = os.environ.get("GITHUB_TOKEN")
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")

# clone once; if the kernel is re-run, update the existing clone in place
if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://{token}@github.com/{REPO}.git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

## **Setup**
Install the kernel's Python deps and pick a GPU-safe render backend, then make the output
folders under /kaggle/working/runs.

In [ ]:
from src.bootstrap import setup
setup(root=ROOT)

In [ ]:
# imports: load every project symbol into the notebook (src/ is on ROOT)
%run $ROOT/src/imports.py
set_seed(cfg.SEED)

## **Fresh start**
Wipe old checkpoints, logs, plots and videos so nothing stale is reloaded.

In [ ]:
import shutil
for d in ("checkpoints", "videos", "plots", "logs"):
    shutil.rmtree(f"{cfg.OUTPUT_ROOT}/{d}", ignore_errors=True)
    os.makedirs(f"{cfg.OUTPUT_ROOT}/{d}", exist_ok=True)
print("cleared old runs on Drive:", cfg.OUTPUT_ROOT)

## **Meet the drone**
A short video of the quadrotor with random (untrained) thrusts - it tumbles and drifts. This is what the four algorithms start from.

In [ ]:
# render the untrained drone in an isolated osmesa subprocess (kernel GL is off for SB3 safety)
path = record_env_video_subprocess(ROOT, f"{cfg.OUTPUT_ROOT}/videos/drone_env.mp4", seconds=6)
if path:
    display(Video(path, embed=True, width=420))

## **Train**
Four algorithms x seeds, each learning from scratch. A run whose log already exists is reloaded.

In [ ]:
runs = []
for algo in cfg.ALGORITHMS:
    for seed in cfg.SEEDS:
        log = f"{cfg.OUTPUT_ROOT}/logs/{algo}_seed{seed}.json"
        if os.path.exists(log):
            runs.append(json.load(open(log)))
        else:
            runs.append(train_one(algo, cfg.CFG, cfg.OUTPUT_ROOT, seed))

## **Results**
The full comparison, mean +/- std over seeds. First an at-a-glance overview panel, then the
headline curves: sample efficiency against model-free SAC, task success, and crash rate while
learning, and final policy quality. Every figure is saved under OUTPUT_ROOT/plots.

In [ ]:
# the whole comparison at a glance: return, success, crash, imagined horizon
plot_overview(runs, f"{cfg.OUTPUT_ROOT}/plots/overview.png");

In [ ]:
# sample efficiency: return vs real steps, with model-free SAC's final return as reference
plot_sample_efficiency(runs, f"{cfg.OUTPUT_ROOT}/plots/sample_efficiency.png");

In [ ]:
# task success while learning: fraction of eval episodes that reach and hold the target
plot_success_rate(runs, f"{cfg.OUTPUT_ROOT}/plots/success_rate.png");

In [ ]:
# stability while learning: fraction of eval episodes that crash
plot_failure_rate(runs, f"{cfg.OUTPUT_ROOT}/plots/crash_rate.png");

In [ ]:
# final policy quality: best evaluation return per algorithm
plot_final_quality(runs, f"{cfg.OUTPUT_ROOT}/plots/final_quality.png");

## **MACURA under the hood**
Why MACURA differs from the others: how far each method trusts the model, MACURA's adaptive
rollout length and trust threshold kappa, its model uncertainty over training, and the spread
of per-rollout truncation lengths.

In [ ]:
# how far each model-based method imagines: MACURA adapts, MBPO ramps, M2AC is fixed
plot_imagined_horizon(runs, f"{cfg.OUTPUT_ROOT}/plots/imagined_horizon.png");

In [ ]:
# MACURA signature: adaptive rollout length grows as kappa (trust threshold) settles
plot_rollout_depth([r for r in runs if r["algo"] == "macura"][0],
                   f"{cfg.OUTPUT_ROOT}/plots/macura_rollout_depth.png");

In [ ]:
# MACURA model uncertainty (first-step GJS) vs the adaptive trust threshold kappa
plot_uncertainty([r for r in runs if r["algo"] == "macura"],
                 f"{cfg.OUTPUT_ROOT}/plots/macura_uncertainty.png");

In [ ]:
# spread of MACURA truncation lengths (a fixed-horizon method would be a single spike)
plot_rollout_distribution([r for r in runs if r["algo"] == "macura"][0],
                          f"{cfg.OUTPUT_ROOT}/plots/macura_rollout_distribution.png");

## **Summary table**
Best return, final return, reached-and-held success rate, and crash rate per algorithm.

In [ ]:
# per-algorithm results table (best/final return, success, crash)
plot_summary_table(runs, f"{cfg.OUTPUT_ROOT}/plots/summary_table.png");

## **Flight demo**
Videos render only if MACURA_RENDER=1 (off by default). The intended way to watch the drone
is live on your Mac after downloading the checkpoints (see Save).

In [ ]:
videos = record_best_videos(cfg.CFG, cfg.OUTPUT_ROOT, device, seeds=cfg.SEEDS, seconds=8) if cfg.RENDER else {}
for name, path in videos.items():
    print(name)
    display(Video(path, embed=True, width=360))

In [ ]:
labels = {name.split("_seed")[0]: path for name, path in videos.items()}
if labels:
    display(Video(stitch_videos_grid(labels, f"{cfg.OUTPUT_ROOT}/videos/reel.mp4", cols=2), embed=True, width=520))

## **Save**
Checkpoints, plots and logs are written under /kaggle/working/runs and saved as the kernel's
Output. Download it with `kaggle kernels output MYUSERNAME/macura-drone -p ./out`, copy the
`checkpoints/` folder to your Mac, and watch the drone fly live:

`mjpython run_live_mac.py --compare runs/checkpoints`

In [ ]:
for r in sorted(runs, key=lambda r: -r["final_eval"]["eval_return"]):
    fe = r["final_eval"]
    print(f'{r["algo"]:7s} seed{r["seed"]}  return {fe["eval_return"]:7.1f}'
          f'  reached {fe["eval_success_rate"]:.2f}  crash {fe["eval_failure_rate"]:.2f}')